# LF6 — degrade_boundary (suy giảm có kiểm soát)

LF6 — degrade_boundary: sinh nhãn hữu dụng bằng suy giảm có kiểm soát.

Ý tưởng (xem docs/LF6_Methodology.md): lấy ảnh ANCHOR (model hạ nguồn đọc đúng
ground-truth tại delta=0 và confidence > TAU_HIGH), suy giảm dần theo từng trục
(mờ / phơi sáng / phân giải / che khuất / cân bằng trắng) tới khi tác vụ thất bại.
Điểm chuyển 1->0 là ĐIỂM GÃY delta*. delta < delta* -> phiếu 1; delta >= delta* ->
phiếu 0; |delta - delta*| < eps -> abstain.

Không rò rỉ: (1) model chấm lại là out-of-fold (ảnh gốc x thuộc fold k -> chấm bằng
M_{-k}); (2) mọi biến thể suy giảm của x thừa hưởng fold của x (chia theo ảnh gốc).

Model hạ nguồn mặc định là hàm giữ chỗ (trả None) -> 0 anchor -> file phiếu rỗng.
Self-test: chạy cell mục 7 bên dưới (không cần model hạ nguồn).

In [ ]:
from __future__ import annotations

import argparse
import csv
import hashlib
import re
from collections import OrderedDict
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Optional

import cv2
import numpy as np
import PIL
from PIL import Image

## 1. Cấu hình

In [ ]:
TASKS = [
    "1_maturity_evaluation",
    "2_foliar_disease",
    "3_trunk_disease",
    "4_crown_disease",
    "5_petiole",
]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

SEED = 42          # tái lập
K = 5              # số fold cross-fitting (chia theo ảnh gốc)
TAU_HIGH = 0.80    # cổng chọn anchor: model đúng tại delta=0 và confidence > TAU_HIGH
EPS_FRAC = 0.10    # cổng biên: |delta - delta*| < EPS_FRAC * (span trục) -> abstain

# Trục suy giảm cho mỗi tác vụ. Độ chín loại phơi sáng/cân bằng trắng: ảnh Roboflow
# đã qua auto-contrast + phơi sáng ×3 khi export (xem docs, mục Thực tế dữ liệu).
_ALL_AXES = ["blur", "exposure", "resolution", "occlusion", "white_balance"]
AXES_BY_TASK = {
    "1_maturity_evaluation": ["blur", "resolution", "occlusion"],
    "2_foliar_disease": _ALL_AXES,
    "3_trunk_disease": _ALL_AXES,
    "4_crown_disease": _ALL_AXES,
    "5_petiole": _ALL_AXES,
}

# Lưới cường độ delta mỗi trục. delta=0.0 là ảnh gốc; delta tăng thì mức suy giảm tăng.
GRIDS = {
    "blur": [0.0, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0],           # bán kính Gaussian (px)
    "exposure": [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0],      # số stop lệch khỏi gốc
    "resolution": [0.0, 0.2, 0.4, 0.6, 0.75, 0.85, 0.9],    # tỉ lệ hạ phân giải
    "occlusion": [0.0, 0.1, 0.2, 0.3, 0.45, 0.6, 0.75],     # tỉ lệ diện tích bị che
    "white_balance": [0.0, 0.1, 0.2, 0.3, 0.45, 0.6, 0.8],  # độ lệch cân bằng trắng
}

# data.yaml Roboflow: names=['dry','green','tender'] -> class 0/1/2
YOLO_STAGE = {0: "dry", 1: "green", 2: "tender"}
# thư mục bệnh (Mendeley) -> tác vụ hạ nguồn
DISEASE_TASK = {
    "Gray Leaf Spot": "2_foliar_disease",
    "Leaf Rot": "2_foliar_disease",
    "Stem Bleeding": "3_trunk_disease",
    "Bud Rot": "4_crown_disease",
    "Bud Root Dropping": "5_petiole",
}


def print_config():
    print("numpy", np.__version__)
    print("Pillow", PIL.__version__)
    print("OpenCV", cv2.__version__)
    print("SEED", SEED)
    print("K", K)
    print("TAU_HIGH", TAU_HIGH)
    print("EPS_FRAC", EPS_FRAC)
    for axis in _ALL_AXES:
        print("grid", axis, GRIDS[axis])
    for task in TASKS:
        print("axes", task, AXES_BY_TASK[task])

In [ ]:
print_config()

## 2. Toán tử suy giảm

In [ ]:
# Mỗi toán tử: D(img, 0.0) == img (đồng nhất) và đơn điệu theo delta.

def _to_arr(img: Image.Image) -> np.ndarray:
    return np.asarray(img.convert("RGB"), dtype=np.float32)


def _to_img(arr: np.ndarray) -> Image.Image:
    clipped = np.clip(arr, 0, 255).astype(np.uint8)
    return Image.fromarray(clipped, "RGB")


def deg_blur(img: Image.Image, delta: float) -> Image.Image:
    if delta <= 0:
        return img
    arr = _to_arr(img)
    k = int(round(delta)) * 2 + 1
    sigma = float(delta)
    out = cv2.GaussianBlur(
        src=arr,
        ksize=(k, k),
        sigmaX=sigma,
    )
    return _to_img(out)


def deg_exposure(img: Image.Image, delta: float) -> Image.Image:
    # giảm |delta| stop: nhân hệ số 2^(-delta)
    if delta <= 0:
        return img
    factor = 2.0 ** (-delta)
    return _to_img(_to_arr(img) * factor)


def deg_resolution(img: Image.Image, delta: float) -> Image.Image:
    # thu nhỏ theo hệ số (1-delta) rồi phóng lại kích thước cũ
    if delta <= 0:
        return img
    w, h = img.size
    f = max(0.02, 1.0 - float(delta))
    small_w = max(1, int(w * f))
    small_h = max(1, int(h * f))
    small = img.resize((small_w, small_h), Image.BILINEAR)
    return small.resize((w, h), Image.BILINEAR).convert("RGB")


def deg_occlusion(img: Image.Image, delta: float) -> Image.Image:
    # phủ khối chữ nhật diện tích = delta * diện tích ảnh, đặt giữa ảnh (tất định)
    if delta <= 0:
        return img
    arr = _to_arr(img)
    h = arr.shape[0]
    w = arr.shape[1]
    frac = min(0.95, float(delta))
    bh = int(round(h * frac ** 0.5))
    bw = int(round(w * frac ** 0.5))
    y0 = (h - bh) // 2
    x0 = (w - bw) // 2
    arr[y0:y0 + bh, x0:x0 + bw, :] = 0.0
    return _to_img(arr)


def deg_white_balance(img: Image.Image, delta: float) -> Image.Image:
    # đẩy kênh R lên, B xuống theo delta
    if delta <= 0:
        return img
    arr = _to_arr(img)
    arr[..., 0] *= (1.0 + float(delta))
    arr[..., 2] *= (1.0 - 0.5 * float(delta))
    return _to_img(arr)


OPERATORS: dict[str, Callable[[Image.Image, float], Image.Image]] = {
    "blur": deg_blur,
    "exposure": deg_exposure,
    "resolution": deg_resolution,
    "occlusion": deg_occlusion,
    "white_balance": deg_white_balance,
}


def degrade(img: Image.Image, axis: str, delta: float) -> Image.Image:
    return OPERATORS[axis](img, delta)

## 3. Model hạ nguồn + `success`

In [ ]:
@dataclass
class DownstreamModels:
    """Giao diện giống 01_label_correctness.ipynb. Mỗi hàm nhận ảnh (PIL, có thể đã
    suy giảm) và trả (dự đoán, confidence). Mặc định trả (None, 0.0) -> 0 anchor.
    predict_maturity -> ('dry'/'green'/'tender' | None, conf); predict_<disease> -> (bool | None, conf).
    """
    predict_maturity: Callable[[Image.Image], tuple] = lambda im: (None, 0.0)
    predict_foliar: Callable[[Image.Image], tuple] = lambda im: (None, 0.0)
    predict_trunk: Callable[[Image.Image], tuple] = lambda im: (None, 0.0)
    predict_crown: Callable[[Image.Image], tuple] = lambda im: (None, 0.0)
    predict_petiole: Callable[[Image.Image], tuple] = lambda im: (None, 0.0)

    def predictor(self, task: str) -> Callable[[Image.Image], tuple]:
        mapping = {
            "1_maturity_evaluation": self.predict_maturity,
            "2_foliar_disease": self.predict_foliar,
            "3_trunk_disease": self.predict_trunk,
            "4_crown_disease": self.predict_crown,
            "5_petiole": self.predict_petiole,
        }
        return mapping[task]


def success(task: str, pred, gt) -> Optional[int]:
    """1 nếu tác vụ đọc đúng GT; 0 nếu sai; None nếu chưa có model.
    gt: độ chín = tập mức thật (list); bệnh = True (ảnh thuộc đúng lớp)."""
    if pred is None:
        return None
    if task == "1_maturity_evaluation":
        if not gt:
            return None
        if pred in gt:
            return 1
        return 0
    if bool(pred):
        return 1
    return 0

## 4. Ảnh gốc & fold cross-fitting

In [ ]:
def original_id(image_id: str, source: str) -> str:
    """Gộp ×3 augment Roboflow về ảnh gốc: '010_jpg.rf.<hash>' -> '010'.
    Ảnh bệnh (Mendeley) không có augment -> chính nó."""
    if source.startswith("coconut-veirf-v5"):
        return re.split(pattern=r"_jpg", string=image_id, maxsplit=1)[0]
    return image_id


def fold_of(orig_id: str, k: int = K, seed: int = SEED) -> int:
    """Gán fold tất định theo ảnh gốc (mọi augment + mọi ảnh suy giảm cùng fold).
    Băm md5 -> không phụ thuộc PYTHONHASHSEED."""
    digest = hashlib.md5(f"{seed}:{orig_id}".encode()).hexdigest()
    return int(digest, 16) % k


# Nhà cung cấp model out-of-fold. Mặc định rỗng -> dùng model truyền vào cho mọi fold.
# Tích hợp thật: register_fold_models({k: model_khong_co_fold_k for k in range(K)}).
_FOLD_MODELS: dict[int, "DownstreamModels"] = {}


def register_fold_models(mapping: dict[int, "DownstreamModels"]):
    global _FOLD_MODELS
    _FOLD_MODELS = dict(mapping)


def models_for_fold(default: "DownstreamModels", k: int) -> "DownstreamModels":
    if k in _FOLD_MODELS:
        return _FOLD_MODELS[k]
    return default

## 5. Điểm gãy trên bao đơn điệu + phiếu λ

In [ ]:
def break_point(success_seq: list, grid: list) -> float:
    """delta* trên bao đơn điệu: s_bar = cummin(success). Trả delta nhỏ nhất mà
    s_bar==0; nếu không bao giờ hỏng -> +inf. Bỏ qua điểm None."""
    s_bar = float("inf")
    for s, d in zip(success_seq, grid):
        if s is None:
            continue
        s_bar = min(s_bar, s)
        if s_bar == 0:
            return float(d)
    return float("inf")


def lf6_vote(delta: float, delta_star: float, eps: float) -> Optional[int]:
    """Phiếu LF6 theo công thức lambda: abstain trong biên eps; 1 nếu nhẹ hơn điểm gãy; 0 nếu nặng hơn."""
    if abs(delta - delta_star) < eps:
        return None
    if delta < delta_star:
        return 1
    return 0

## 6. Chọn anchor, dò điểm gãy, ghi manifest

In [ ]:
def load_gt(root: Path):
    """Đọc GT giống notebook correctness. Trả {image_id: (path, source, task, gt)}.
    gt = list mức chín (độ chín) hoặc True (ảnh bệnh). Fail-fast nếu thiếu thư mục nguồn."""
    gt = OrderedDict()
    rb = root / "Dataset" / "coconut-veirf-v5"
    if not rb.exists():
        raise SystemExit(f"Không thấy {rb} — kiểm tra lại đường dẫn Dataset/coconut-veirf-v5.")
    for split in ("train", "valid", "test"):
        img_dir = rb / split / "images"
        lbl_dir = rb / split / "labels"
        if not img_dir.exists():
            raise SystemExit(f"Không thấy {img_dir} — kiểm tra lại cấu trúc bộ Roboflow.")
        for img in sorted(img_dir.iterdir()):
            if img.suffix.lower() not in IMG_EXTS:
                continue
            txt = lbl_dir / (img.stem + ".txt")
            stages = []
            if txt.exists():
                for line in txt.read_text().splitlines():
                    if line.strip():
                        cls = int(line.split()[0])
                        stages.append(YOLO_STAGE.get(cls, "?"))
            source = f"coconut-veirf-v5/{split}"
            gt[img.stem] = (img, source, "1_maturity_evaluation", stages)
    db = root / "Dataset" / "Coconut Tree Disease Dataset"
    if not db.exists():
        raise SystemExit(f"Không thấy {db} — kiểm tra lại đường dẫn Coconut Tree Disease Dataset.")
    for folder in sorted(db.iterdir()):
        if not folder.is_dir():
            continue
        task = DISEASE_TASK.get(folder.name)
        if task is None:
            raise SystemExit(f"Thư mục bệnh lạ: {folder.name} — cập nhật DISEASE_TASK.")
        for img in sorted(folder.rglob("*")):
            if img.suffix.lower() in IMG_EXTS:
                gt[img.stem] = (img, f"disease/{folder.name}", task, True)
    return gt


def select_anchors(models: "DownstreamModels", gt, tau_high: float = TAU_HIGH):
    """Anchor = ảnh model đọc đúng tại delta=0 và confidence > tau_high (dùng model out-of-fold).
    Trả list (image_id, path, source, task, gt, fold)."""
    anchors = []
    for image_id, (path, source, task, g) in gt.items():
        k = fold_of(original_id(image_id, source))
        model = models_for_fold(models, k)
        try:
            img = Image.open(path)
        except Exception:
            continue
        pred, conf = model.predictor(task)(img)
        if success(task, pred, g) == 1 and conf > tau_high:
            anchors.append((image_id, path, source, task, g, k))
    return anchors


def build(
    root: Path,
    models: "DownstreamModels",
    out: Path,
    tau_high: float = TAU_HIGH,
    eps_frac: float = EPS_FRAC,
):
    """Chạy LF6 -> ghi labels/votes/lf6_degradation.csv (schema chung, long)."""
    gt = load_gt(root)
    anchors = select_anchors(models, gt, tau_high)
    rows = []
    for image_id, path, source, task, g, k in anchors:
        model = models_for_fold(models, k)
        base = Image.open(path)
        orig = original_id(image_id, source)
        for axis in AXES_BY_TASK[task]:
            grid = GRIDS[axis]
            span = grid[-1] - grid[0]
            eps = eps_frac * span
            seq = []
            for d in grid:
                pred, _ = model.predictor(task)(degrade(base, axis, d))
                seq.append(success(task, pred, g))
            dstar = break_point(seq, grid)
            for d in grid:
                vote = lf6_vote(d, dstar, eps)
                synth_id = image_id + "__" + axis + "__" + str(d)
                rows.append(make_vote(
                    lf="lf6_degradation",
                    image_id=synth_id,
                    task=task,
                    vote=vote,
                    source=source,
                    path=str(path),
                    base_image=image_id,
                    original_id=orig,
                    fold=k,
                    axis=axis,
                    delta=d,
                    delta_star=dstar,
                ))
    extra = ["base_image", "original_id", "fold", "axis", "delta", "delta_star"]
    write_lf_votes(out, rows, extra_fields=extra)
    print("anchor", len(anchors))
    clean = [r for r in rows if r is not None]
    return clean


## 7. Self-test

In [ ]:
def _selftest():
    """Kiểm chứng không cần model hạ nguồn: (1) toán tử đồng nhất tại 0 & đơn điệu;
    (2) break_point khôi phục điểm gãy từ chuỗi nhiễu nhờ cummin; (3) fold tất định;
    (4) lf6_vote khớp công thức; (5) đầu-cuối với model giả đơn điệu theo độ nét."""
    rng = np.random.default_rng(0)
    arr0 = rng.integers(0, 256, size=(64, 64, 3))
    img = _to_img(arr0.astype(np.float32))
    a0 = _to_arr(img)

    for axis in _ALL_AXES:
        same = np.array_equal(_to_arr(degrade(img, axis, 0.0)), a0)
        assert same, f"{axis}: delta=0 phải đồng nhất"
        diffs = []
        for d in GRIDS[axis]:
            diffs.append(float(np.abs(_to_arr(degrade(img, axis, d)) - a0).mean()))
        for i in range(len(diffs) - 1):
            assert diffs[i] <= diffs[i + 1] + 1e-6, f"{axis}: mức méo phải đơn điệu: {diffs}"

    grid = GRIDS["blur"]
    noisy = [1, 1, 1, 0, 1, 0, 0]
    assert break_point(noisy, grid) == 3.0
    assert break_point([1, 1, 1, 1, 1, 1, 1], grid) == float("inf")
    assert break_point([1, 1, None, 0, 0, 0, 0], grid) == 3.0

    assert fold_of("010") == fold_of("010")
    assert fold_of(original_id("010_jpg.rf.abc", "coconut-veirf-v5/train")) == fold_of("010")
    assert fold_of(original_id("010_jpg.rf.xyz", "coconut-veirf-v5/train")) == fold_of("010")

    assert lf6_vote(2.0, 5.0, 1.0) == 1
    assert lf6_vote(7.0, 5.0, 1.0) == 0
    assert lf6_vote(5.0, 5.0, 1.0) is None
    assert lf6_vote(4.5, 5.0, 1.0) is None

    edges = np.zeros((64, 64, 3), np.float32)
    edges[:, ::4, :] = 255
    anchor = _to_img(edges)

    def sharp(im):
        gray = cv2.cvtColor(_to_arr(im).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        return float(cv2.Laplacian(gray, cv2.CV_64F).var())

    thr = sharp(anchor) * 0.25

    def fake_foliar(im):
        return (sharp(im) >= thr, 0.99)

    seq = []
    for d in GRIDS["blur"]:
        pred, _ = fake_foliar(degrade(anchor, "blur", d))
        seq.append(success("2_foliar_disease", pred, True))
    dstar = break_point(seq, GRIDS["blur"])
    assert 0 < dstar < float("inf"), f"model giả phải gãy khi mờ tăng: {seq}"
    span = GRIDS["blur"][-1] - GRIDS["blur"][0]
    votes = []
    for d in GRIDS["blur"]:
        votes.append(lf6_vote(d, dstar, EPS_FRAC * span))
    assert 1 in votes
    assert 0 in votes

    print("SELF-TEST OK — toan tu don dieu, diem gay dung, fold tat dinh, phieu khop cong thuc.")

In [ ]:
_selftest()

## 8. Chạy

In [ ]:
def repo_root() -> Path:
    """Thư mục gốc repo (chứa Dataset/). Chạy từ gốc repo hoặc từ notebooks/. Fail-fast."""
    cwd = Path.cwd()
    if (cwd / "Dataset").exists():
        return cwd
    if (cwd.parent / "Dataset").exists():
        return cwd.parent
    raise SystemExit("Không thấy thư mục Dataset/ — chạy từ thư mục gốc repo hoặc notebooks/.")

In [ ]:
root = repo_root()
print("ROOT", root)

# Nap module dung chung (schema phieu + writer). Khong import duoc .ipynb -> dung %run.
UTILS = root / "src" / "utils" / "lf_io.ipynb"
if not UTILS.exists():
    raise SystemExit("Khong thay " + str(UTILS))
get_ipython().run_line_magic("run", str(UTILS))
# Tích hợp model thật:
#   register_fold_models({k: model_khong_co_fold_k for k in range(K)})
#   models = DownstreamModels(predict_maturity=..., predict_foliar=..., ...)
models = DownstreamModels()
rows = build(root, models, root / "labels" / "votes" / "lf6_degradation.csv")